In [14]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, round as spark_round

spark = SparkSession.builder.appName("Tugas6").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

A. EXtract

In [16]:
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

for nama, d in [("Transaksi", df_trx), ("Produk", df_produk), ("Ulasan", df_ulasan)]:
    print(nama, ":", d.count(), "baris")
    d.printSchema()

Transaksi : 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Produk : 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Ulasan : 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



B. Join

In [22]:
df_join = df_trx.join(df_produk, on="product_id", how="inner")
df_join = df_join.join(df_ulasan, on="order_id", how="left")
df_join = df_join.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))
df_join.show(5)
print("Jumlah baris setelah join:", df_join.count())

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|  NULL|         1500000|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|  NULL|          450000|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
o

C. Data kosong

In [18]:
df_join = df_join.withColumn("ada_ulasan", col("rating").isNotNull())
df_final = df_join.na.fill({"rating": 0})
df_final.select("order_id", "kategori", "rating", "ada_ulasan").show(10)
print("Sisa null di rating:", df_final.filter(col("rating").isNull()).count())

+--------+------------+------+----------+
|order_id|    kategori|rating|ada_ulasan|
+--------+------------+------+----------+
|     TX0|  Elektronik|     1|      true|
|     TX1|     Fashion|     0|     false|
|     TX2|  Elektronik|     2|      true|
|     TX3|     Fashion|     5|      true|
|     TX4|Rumah Tangga|     0|     false|
|     TX5|  Elektronik|     5|      true|
|     TX6|   Kesehatan|     4|      true|
|     TX7|     Makanan|     0|     false|
|     TX8|  Elektronik|     5|      true|
|     TX9|  Elektronik|     0|     false|
+--------+------------+------+----------+
only showing top 10 rows

Sisa null di rating: 0


Rating valid cuma 1 sampai 5, jadi 0 ada di luar skala dan langsung kebaca sebagai "belum ada ulasan", ga ketuker sama rating asli. Kalau diisi 3 atau rata-rata, datanya seolah-olah ada yang ngasih rating padahal ga ada. Catatan pas hitung rata-rata rating: filter rating lebih dari 0 dulu biar ga ketarik turun.

D. Load ke HDFS

In [19]:
!hdfs dfs -mkdir -p /user/pathan/tugas6

df_final.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/pathan/tugas6/hasil_etl"
)

!hdfs dfs -ls -R /user/pathan/tugas6/hasil_etl

-rw-r--r--   3 pathan supergroup          0 2026-10-01 06:39 /user/pathan/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - pathan supergroup          0 2026-10-01 06:39 /user/pathan/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 pathan supergroup      16364 2026-10-01 06:39 /user/pathan/tugas6/hasil_etl/kategori=Elektronik/part-00000-57e64441-24eb-4370-840c-8c2da4ead014.c000.snappy.parquet
drwxr-xr-x   - pathan supergroup          0 2026-10-01 06:39 /user/pathan/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 pathan supergroup      11134 2026-10-01 06:39 /user/pathan/tugas6/hasil_etl/kategori=Fashion/part-00000-57e64441-24eb-4370-840c-8c2da4ead014.c000.snappy.parquet
drwxr-xr-x   - pathan supergroup          0 2026-10-01 06:39 /user/pathan/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 pathan supergroup      10393 2026-10-01 06:39 /user/pathan/tugas6/hasil_etl/kategori=Kesehatan/part-00000-57e64441-24eb-4370-840c-8c2da4ead014.c000.snappy.parquet
drwxr-xr-x   - pathan supergroup      

In [20]:
df_cek = spark.read.parquet("hdfs://localhost:9000/user/pathan/tugas6/hasil_etl")
print("Jumlah baris di HDFS:", df_cek.count())
df_cek.printSchema()

Jumlah baris di HDFS: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- harga: long (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- total_pendapatan: long (nullable = true)
 |-- ada_ulasan: boolean (nullable = true)
 |-- kategori: string (nullable = true)



E. Insight

In [21]:
df_insight = df_cek.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    count(when(col("ada_ulasan") == True, 1)).alias("transaksi_berulasan")
).withColumn(
    "persen_ulasan", spark_round(col("transaksi_berulasan") / col("total_transaksi") * 100, 2)
).orderBy("persen_ulasan")

df_insight.show()

+------------+---------------+-------------------+-------------+
|    kategori|total_transaksi|transaksi_berulasan|persen_ulasan|
+------------+---------------+-------------------+-------------+
|     Makanan|            536|                369|        68.84|
|   Kesehatan|            961|                662|        68.89|
|     Fashion|           1035|                726|        70.14|
|Rumah Tangga|            815|                575|        70.55|
|  Elektronik|           1653|               1168|        70.66|
+------------+---------------+-------------------+-------------+



Kategori dengan persentase ulasan paling rendah adalah Makanan dengan 68.84% transaksi berulasan. Selisihnya tipis dari kategori lain (68.89% sampai 70.66%), jadi masalahnya ada di semua kategori, bukan cuma satu. Buat tim marketing, sekitar 30% transaksi ga punya ulasan, dan kategori Makanan paling layak dikejar duluan lewat reminder minta ulasan setelah pesanan sampai atau voucher buat yang ngisi, karena ulasan ngaruh ke keputusan beli pelanggan lain.